eid data source: https://www.pxweb.bfs.admin.ch/pxweb/fr/px-x-1703030000_100/-/px-x-1703030000_100.px/table/tableViewLayout2/ $\newline$
2025 population data source: https://www.pxweb.bfs.admin.ch/pxweb/fr/px-x-0102020000_202/-/px-x-0102020000_202.px/
$\newline$
Note: this data analysis was done without any help from AI.


In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import statsmodels.api as smf

In [311]:
cantons_df = pd.read_csv("cantons_data.csv")
eid_df = pd.read_csv("eid_votation.csv")

In [312]:
cantons_df = cantons_df[cantons_df["Sex"] != "Sexe - total"]
cantons_df = cantons_df[cantons_df["Age"] != "100 ans ou plus"]
cantons_df = cantons_df[cantons_df["Canton"] != "Suisse"]
cantons_df = cantons_df[cantons_df["Canton"] != "Sans indication"]
col = ["Canton", "Age", "Sex", "Count", "Immigration", "Swiss citizenship acquisition"]
cantons_df = cantons_df[col]

cantons_df.head()

,Canton,Age,Sex,Count,Immigration,Swiss citizenship acquisition
332,Z�rich,18 ans,Homme,6227,29,106
333,Z�rich,19 ans,Homme,5993,34,74
334,Z�rich,20 ans,Homme,6065,39,63
335,Z�rich,21 ans,Homme,5925,32,42
336,Z�rich,22 ans,Homme,5819,38,28


In [313]:
cantons_df["Canton"].unique()

array(['Z�rich', 'Bern / Berne', 'Luzern', 'Uri', 'Schwyz', 'Obwalden',
       'Nidwalden', 'Glarus', 'Zug', 'Fribourg / Freiburg', 'Solothurn',
       'Basel-Stadt', 'Basel-Landschaft', 'Schaffhausen',
       'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'St. Gallen',
       'Graub�nden / Grigioni / Grischun', 'Aargau', 'Thurgau', 'Ticino',
       'Vaud', 'Valais / Wallis', 'Neuch�tel', 'Gen�ve', 'Jura'],
      dtype=object)

In [314]:
def fix_cantons(cantons_df):
    cantons_df = cantons_df.replace('Z�rich','Zurich')
    cantons_df = cantons_df.replace('Bern / Berne','Berne')
    cantons_df = cantons_df.replace('Fribourg / Freiburg','Fribourg')
    cantons_df = cantons_df.replace('Graub�nden / Grigioni / Grischun','Grischun')
    cantons_df = cantons_df.replace('Valais / Wallis','Valais')
    cantons_df = cantons_df.replace('Neuch�tel','Neuchatel')
    cantons_df = cantons_df.replace('Gen�ve','Geneve')

    return cantons_df

cantons_df = fix_cantons(cantons_df)
eid_df = fix_cantons(eid_df)



In [315]:
cantons_df["Canton"].unique()

array(['Zurich', 'Berne', 'Luzern', 'Uri', 'Schwyz', 'Obwalden',
       'Nidwalden', 'Glarus', 'Zug', 'Fribourg', 'Solothurn',
       'Basel-Stadt', 'Basel-Landschaft', 'Schaffhausen',
       'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'St. Gallen',
       'Grischun', 'Aargau', 'Thurgau', 'Ticino', 'Vaud', 'Valais',
       'Neuchatel', 'Geneve', 'Jura'], dtype=object)

In [316]:
eid_df.head()

,Canton,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,Yes %
0,Suisse,5641040,2796897,49.58,2747948,1384586,1363362,50.39
1,Zurich,976490,509297,52.16,503923,274702,229221,54.51
2,Berne,750745,358855,47.80,354158,171771,182387,48.50
3,Luzern,288363,153604,53.27,151603,80261,71342,52.94
4,Uri,27226,12731,46.76,12567,5109,7458,40.65


In [ ]:
cantons_df.head()

,Canton,Age,Sex,Count,Immigration,Swiss citizenship acquisition
332,Zurich,18 ans,Homme,6227,29,106
333,Zurich,19 ans,Homme,5993,34,74
334,Zurich,20 ans,Homme,6065,39,63
335,Zurich,21 ans,Homme,5925,32,42
336,Zurich,22 ans,Homme,5819,38,28


In [ ]:
# for later graphs that have to do with age groups

age_groups = [range(18, 24), range(24,30), range(30, 36), range(36, 42), range(42, 48), range(48, 54), range(54, 60), range(60, 66),
              range(66, 72), range(72, 78), range(78, 84), range(84, 90), range(90, 96)]
def age_sort(df, age_groups):
    for i in range(95, 100):
        df = df[df["Age"] != str(i) + " ans"]
    # group by ages to count population groups
    for age_group in age_groups:
        new_string = str(age_group[0]) + " - " + str(age_group[-1])
        for age in age_group:
            age_string = str(age) + " ans"
            df = df.replace(age_string, new_string)
    df.reset_index()
    return df
        

In [ ]:
def age_transform(df):
    for i in range(18, 100):
        age_string = str(i) + " ans"
        df = df.replace(age_string, i)
    return df

In [319]:
# represent demographics with percentages instead of count
def canton_proportions(df):
    df["Count"] = df["Count"] / df.groupby("Canton")["Count"].transform('sum')
    return df

In [320]:
cantons_df = canton_proportions(cantons_df)
cantons_df = cantons_df.rename(columns={"Count":"Proportion"})
cantons_df = cantons_df.replace("Homme","Male")
cantons_df = cantons_df.replace("Femme","Female")
cantons_df.head()

,Canton,Age,Sex,Proportion,Immigration,Swiss citizenship acquisition
332,Zurich,18 ans,Male,0.006559,29,106
333,Zurich,19 ans,Male,0.006313,34,74
334,Zurich,20 ans,Male,0.006389,39,63
335,Zurich,21 ans,Male,0.006241,32,42
336,Zurich,22 ans,Male,0.006129,38,28


In [ ]:
cantons_df = age_sort(cantons_df, age_groups)
cantons_df = cantons_df.groupby(["Canton", "Age", "Sex"]).sum()
cantons_df.head(10)

Proportion  Immigration  Swiss citizenship acquisition
Canton Age     Sex                                                           
Aargau 18 - 23 Female    0.035195           53                            130
               Male      0.037526           52                            123
       24 - 29 Female    0.037362           76                             89
               Male      0.039086           71                             37
       30 - 35 Female    0.042295           57                            123
               Male      0.041897           79                             88
       36 - 41 Female    0.044499           50                            224
               Male      0.044360           57                            213
       42 - 47 Female    0.045031           35                            228
               Male      0.044403           50                            208

In [322]:
cantons_df = cantons_df.reset_index()
cantons_df.pivot(columns=["Age", "Sex"], index="Canton", values=["Proportion"])

Proportion                                          \
Age                       18 - 23             24 - 29             30 - 35   
Sex                        Female      Male    Female      Male    Female   
Canton                                                                      
Aargau                   0.035195  0.037526  0.037362  0.039086  0.042295   
Appenzell Ausserrhoden   0.032458  0.034014  0.033302  0.036835  0.038470   
Appenzell Innerrhoden    0.036391  0.036558  0.040230  0.044821  0.042985   
Basel-Landschaft         0.033515  0.034915  0.034461  0.036438  0.036480   
Basel-Stadt              0.032536  0.033356  0.041357  0.040565  0.048737   
Berne                    0.033202  0.034375  0.036659  0.036731  0.042369   
Fribourg                 0.039032  0.040677  0.043477  0.044365  0.046958   
Geneve                   0.045883  0.046939  0.046998  0.047323  0.044977   
Glarus                   0.030969  0.034462  0.035114  0.037071  0.038414   
Grischun                 0.031511  0.032252  0.035037  0.037794  0.037218   
Jura                     0.038035  0.040796  0.038726  0.043825  0.039032   
Luzern                   0.035619  0.036902  0.042016  0.042681  0.046135   
Neuchatel                0.041589  0.043874  0.040818  0.043791  0.041075   
Nidwalden                0.030143  0.034463  0.030427  0.035819  0.038405   
Obwalden                 0.035665  0.035774  0.033838  0.037748  0.038625   
Schaffhausen             0.032802  0.033632  0.034989  0.039363  0.036460   
Schwyz                   0.033277  0.035397  0.036002  0.038168  0.039432   
Solothurn                0.032488  0.034066  0.036349  0.037270  0.040503   
St. Gallen               0.037094  0.038589  0.041237  0.042976  0.043283   
Thurgau                  0.035569  0.036818  0.036123  0.039138  0.042074   
Ticino                   0.036371  0.038453  0.036778  0.038287  0.034317   
Uri                      0.035940  0.038198  0.036199  0.041974  0.039309   
Valais                   0.033850  0.036458  0.038971  0.039959  0.041269   
Vaud                     0.042447  0.044596  0.044358  0.043928  0.045681   
Zug                      0.035169  0.036657  0.036054  0.036336  0.040118   
Zurich                   0.036394  0.037786  0.039810  0.039636  0.046078   

                                                                          ...  \
Age                                36 - 41             42 - 47            ...   
Sex                         Male    Female      Male    Female      Male  ...   
Canton                                                                    ...   
Aargau                  0.041897  0.044499  0.044360  0.045031  0.044403  ...   
Appenzell Ausserrhoden  0.041792  0.045088  0.044376  0.043585  0.044481  ...   
Appenzell Innerrhoden   0.048911  0.042567  0.044070  0.038227  0.043235  ...   
Basel-Landschaft        0.034915  0.039313  0.037896  0.040655  0.038018  ...   
Basel-Stadt             0.048318  0.045047  0.044332  0.041700  0.040909  ...   
Berne                   0.041805  0.044037  0.043042  0.043897  0.042223  ...   
Fribourg                0.046779  0.044299  0.043679  0.043486  0.042203  ...   
Geneve                  0.044640  0.041993  0.038907  0.043928  0.039695  ...   
Glarus                  0.040870  0.042866  0.045667  0.042022  0.043480  ...   
Grischun                0.038327  0.039709  0.040212  0.039615  0.039442  ...   
Jura                    0.042521  0.039262  0.040374  0.039109  0.037997  ...   
Luzern                  0.046466  0.045470  0.045681  0.045147  0.043297  ...   
Neuchatel               0.042112  0.040580  0.037818  0.041029  0.037626  ...   
Nidwalden               0.040864  0.040202  0.043985  0.039414  0.039571  ...   
Obwalden                0.039721  0.043813  0.045567  0.044764  0.041475  ...   
Schaffhausen            0.038458  0.040739  0.041455  0.040701  0.041154  ...   
Schwyz                  0.042789  0.042278  0.043607  0.042529  0.044518  ...   
Solothurn  

In [324]:
eid_df = eid_df.groupby(["Canton"]).sum()
eid_df.head(10)

,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,Yes %
Canton,,,,,,,
Aargau,447798,232292,51.87,230470,113121,117349,49.08
Appenzell Ausserrhoden,39526,21896,55.40,21652,9949,11703,45.95
Appenzell Innerrhoden,12385,6006,48.49,5859,2644,3215,45.13
Basel-Landschaft,192244,96303,50.09,93940,46245,47695,49.23
Basel-Stadt,114378,55409,48.44,54544,30976,23568,56.79
Berne,750745,358855,47.80,354158,171771,182387,48.50
Fribourg,219163,100846,46.01,98853,49792,49061,50.37
Geneve,285837,119086,41.66,112906,62308,50598,55.19
Glarus,26836,12159,45.31,12008,4965,7043,41.35


In [327]:
data_df = pd.merge(left=eid_df, right=cantons_df, left_on="Canton", right_on="Canton", how="inner")
data_df

,Canton,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,Yes %,Age,Sex,Proportion,Immigration,Swiss citizenship acquisition
0,Aargau,447798,232292,51.87,230470,113121,117349,49.08,18 - 23,Female,0.035195,53,130
1,Aargau,447798,232292,51.87,230470,113121,117349,49.08,18 - 23,Male,0.037526,52,123
2,Aargau,447798,232292,51.87,230470,113121,117349,49.08,24 - 29,Female,0.037362,76,89
3,Aargau,447798,232292,51.87,230470,113121,117349,49.08,24 - 29,Male,0.039086,71,37
4,Aargau,447798,232292,51.87,230470,113121,117349,49.08,30 - 35,Female,0.042295,57,123
...,...,...,...,...,...,...,...,...,...,...,...,...,...
671,Zurich,976490,509297,52.16,503923,274702,229221,54.51,78 - 83,Male,0.026730,34,4
672,Zurich,976490,509297,52.16,503923,274702,229221,54.51,84 - 89,Female,0.021014,18,2
673,Zurich,976490,509297,52.16,503923,274702,229221,54.51,84 - 89,Male,0.013972,14,0
674,Zurich,976490,509297,52.16,503923,274702,229221,54.51,90 - 95,Female,0.009352,2,0


In [ ]:
smf.ols(formula='Yes % ~ Age', data=data_df).fit()